# Anomaly Detection in CT scans using Feature Extraction and XGBoost

### Create the Dataset class and load the dataset

In [10]:
from torch.utils.data import Dataset
from torchvision.io import decode_image
import os
import pandas as pd

class ImageDataset(Dataset):
    def __init__(self, annotation_file, image_dir, transform=None):
        self.img_labels = pd.read_csv(annotation_file)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, "{0:03}".format(self.img_labels.iloc[idx, 0]))
        img_path = img_path + '.png'
        image = decode_image(img_path)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        return image, label

In [11]:
import torchvision
import torch

def convert_to_3channels(x):
    """Convert image to 3-channel RGB"""
    if x.shape[0] == 1:
        # Grayscale to RGB
        return x.repeat(3, 1, 1)
    elif x.shape[0] == 4:
        # RGBA to RGB (drop alpha channel)
        return x[:3, :, :]
    else:
        return x

transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ConvertImageDtype(torch.float),
    torchvision.transforms.Lambda(convert_to_3channels),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
    

dataset =ImageDataset(annotation_file='labels.csv', image_dir='head_ct/head_ct', transform=transform)

img, label = dataset[0]
print(f"Image shape: {img.shape}, Label: {label}")

Image shape: torch.Size([3, 224, 224]), Label: 1


### Train Val Test Split and DataLoaders

In [12]:
from torch.utils.data import random_split

trainsize = int(0.8 * len(dataset))
validsize = int(0.1 * len(dataset))
testsize = len(dataset) - trainsize - validsize
train_dataset, valid_dataset, test_dataset = random_split(dataset, [trainsize, validsize, testsize])

In [13]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=32, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

### Feature Extractor initialization

In [14]:
from torchvision.models import resnet18
from torch import nn

model = resnet18(weights='IMAGENET1K_V1')

feature_extractor = nn.Sequential(
    model.conv1,
    model.bn1,
    model.relu,
    model.maxpool,
    model.layer1,
    model.layer2,
    nn.AdaptiveAvgPool2d((1, 1))
)

### Feature Extraction

In [ ]:
import numpy as np

training_features = []
validation_features = []
test_features = []
train_labels = []
validation_labels = []
test_labels = []

with torch.no_grad():
    for batch in train_loader:
        input_tensor, labels = batch
        print(input_tensor.shape)
        features = feature_extractor(input_tensor)
        print(f"Extracted features shape: {features.shape}")
        features = features.view(features.size(0), -1)
        features = features.detach().numpy()
        training_features.append(features)
        train_labels.append(labels)

    for batch in valid_loader:
        input_tensor, labels = batch
        print(input_tensor.shape)
        features = feature_extractor(input_tensor)
        print(f"Extracted features shape: {features.shape}")
        features = features.view(features.size(0), -1)
        features = features.detach().numpy()
        validation_features.append(features)
        validation_labels.append(labels)

    for batch in test_loader:
        input_tensor, labels = batch
        print(input_tensor.shape)
        features = feature_extractor(input_tensor)
        print(f"Extracted features shape: {features.shape}")
        features = features.view(features.size(0), -1)
        features = features.detach().numpy()
        test_features.append(features)
        test_labels.append(labels)

training_features = np.concatenate(training_features)
validation_features = np.concatenate(validation_features)
test_features = np.concatenate(test_features)
train_labels = torch.cat(train_labels).numpy()
validation_labels = torch.cat(validation_labels).numpy()
test_labels = torch.cat(test_labels).numpy()

print(f"Training features shape: {training_features.shape}")
print(f"Validation features shape: {validation_features.shape}")
print(f"Test features shape: {test_features.shape}")

torch.Size([32, 3, 224, 224])
Extracted features shape: torch.Size([32, 128, 1, 1])
torch.Size([32, 3, 224, 224])
Extracted features shape: torch.Size([32, 128, 1, 1])
torch.Size([32, 3, 224, 224])
Extracted features shape: torch.Size([32, 128, 1, 1])
torch.Size([32, 3, 224, 224])
Extracted features shape: torch.Size([32, 128, 1, 1])
torch.Size([32, 3, 224, 224])
Extracted features shape: torch.Size([32, 128, 1, 1])
torch.Size([20, 3, 224, 224])
Extracted features shape: torch.Size([20, 128, 1, 1])
torch.Size([20, 3, 224, 224])
Extracted features shape: torch.Size([20, 128, 1, 1])
Training features shape: (160, 128)
Validation features shape: (20, 128)
Test features shape: (20, 128)


### XGBoost preparation and training

In [42]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'gamma': [0, 1, 5],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2],
    'min_child_weight': [1, 3, 5],
    'scale_pos_weight': [1, 2, 5]
    }

xgb_clf = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', early_stopping_rounds=10)
random_search = RandomizedSearchCV(estimator=xgb_clf, param_distributions=param_dist, n_iter=20, cv=3, verbose=False, refit=True)
random_search.fit(training_features, train_labels, eval_set=[(validation_features, validation_labels)], verbose=False)


best_xgb = random_search.best_estimator_
training_preds = best_xgb.predict(training_features)
training_accuracy = accuracy_score(train_labels, training_preds)
print(f"Training Accuracy: {training_accuracy:.4f}")
validation_preds = best_xgb.predict(validation_features)
accuracy = accuracy_score(validation_labels, validation_preds)
print(f"Validation Accuracy: {accuracy:.4f}")






Training Accuracy: 0.9875
Validation Accuracy: 0.8500


In [43]:
from sklearn.metrics import classification_report

print("Classification Report:")
print(classification_report(validation_labels, validation_preds))

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.82      0.86        11
           1       0.80      0.89      0.84         9

    accuracy                           0.85        20
   macro avg       0.85      0.85      0.85        20
weighted avg       0.86      0.85      0.85        20



### Export Feature Extractor, XGBoost Model and Test Data

In [44]:
best_xgb.save_model("xgb_model.ubj")
torch.save(feature_extractor.state_dict(), "feature_extractor.pth")

np.savetxt("training_features.csv", training_features, delimiter=",")
np.savetxt("training_labels.csv", train_labels, delimiter=",")
np.savetxt("validation_features.csv", validation_features, delimiter=",")
np.savetxt("validation_labels.csv", validation_labels, delimiter=",")
np.savetxt("test_features.csv", test_features, delimiter=",")
np.savetxt("test_labels.csv", test_labels, delimiter=",")